# Expanding window temporal modeling

The experiment trains on flights 1 through 40 and tests on 41 through 50, then expands the training history by ten launches until flights 81 through 90. Every preprocessing step is fitted inside its training window.

In [ ]:
from spacex_falcon.analytics import modeling_frame
from spacex_falcon.data import load_launch_data
from spacex_falcon.evaluation import run_experiments

data = modeling_frame(load_launch_data())
results = run_experiments(data)

## Data quality and target shift

The validated dataset contains 90 ordered launches. The final ten launches are all successful, while earlier windows contain both outcomes. That target shift makes aggregate accuracy optimistic and makes a random split answer the wrong forecasting question.

In [ ]:
window_balance = (data.assign(Window=((data['FlightNumber'] - 1) // 10) + 1).groupby('Window')['Class'].agg(['count', 'mean']))
window_balance

## Feature reasoning

The deployed feature vector contains only values known before launch and explicitly supplied in the interface: flight number, payload mass, booster prior flights, orbit, launch site, grid fins, reuse status, and landing legs. Landing outcome text and landing pad are excluded.

## Baselines and model comparison

Balanced accuracy is the primary comparison because later windows contain far more successes than failures. ROC AUC, Brier score, log loss, and F1 provide complementary views. Bootstrap intervals prevent small numerical differences from being overstated.

In [ ]:
results.comparison[['Model', 'BalancedAccuracy', 'BalancedAccuracyCILow', 'BalancedAccuracyCIHigh', 'ROCAUC', 'Brier', 'F1']]

## Feature ablation

The time only model tests whether technological maturation alone explains the apparent predictability. Mission, booster, and full feature sets show the incremental information contributed by each group.

In [ ]:
results.ablations[['FeatureSet', 'BalancedAccuracy', 'ROCAUC', 'Brier']]

## Results

Random forest reaches 0.689 aggregate balanced accuracy, compared with 0.500 for the historical rate baseline. Its 95% bootstrap interval spans 0.537 to 0.852, so the apparent lead is not precise. Logistic regression reaches 0.669 and provides the best Brier score at 0.150.

## Calibration

Sigmoid calibration is fitted with temporal splits inside each training window. It changes Brier score from 0.150 to 0.178. Because squared probability error becomes worse, the raw logistic probability is retained.

In [ ]:
results.calibration[['Model', 'BalancedAccuracy', 'ROCAUC', 'Brier', 'LogLoss']]

## Error analysis and interpretation

Errors should be reviewed by orbit, launch site, reuse, payload, and historical window. Coefficients and feature importance are descriptive because site and booster configuration are confounded with program era.

In [ ]:
from spacex_falcon.evaluation import error_analysis
error_analysis(data, results.predictions, 'Logistic regression')

## Limitations

The sample is small, the final window contains no failures, historical era is strongly confounded with hardware and site, and bootstrap intervals do not capture uncertainty in how the course dataset was constructed. Results are retrospective and not operational.

## Conclusion

Chronological evaluation is materially less flattering than the original random split, but it is more defensible. Mission and booster features add signal beyond time alone, yet uncertainty remains large enough that model rankings should be treated as exploratory rather than definitive.